# 03b - Model Registration: Service Orders (Tabular)

**Azure ML MLOps Workshop | Track B - Tabular Data**

### Context

In Notebook 03, you registered the best text classifier into the Model Registry.
Now you'll register the best **tabular classifier** — so the registry holds
two models side by side, just like a real team would manage.

### What you'll do:
1. Find the best run from the repair classifier experiments
2. Register it as `contoso-repair-classifier`
3. Browse the registry — see both models coexisting

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow
from mlflow.tracking import MlflowClient

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

workspace = ml_client.workspaces.get(WORKSPACE_NAME)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
mlflow_client = MlflowClient()
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Find the Best Experiment Run

In [ ]:
EXPERIMENT_NAME = "contoso-repair-classifier"
experiment = mlflow_client.get_experiment_by_name(EXPERIMENT_NAME)

all_runs = mlflow_client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=20,
)

finished_runs = [r for r in all_runs if r.info.status == "FINISHED" and "f1" in r.data.metrics]
finished_runs.sort(key=lambda r: r.data.metrics.get("f1", 0), reverse=True)
best_run = finished_runs[0]

print(f"Best run:")
print(f"  Run ID:     {best_run.info.run_id}")
print(f"  Model type: {best_run.data.params.get('model_type', 'N/A')}")
print(f"  F1 score:   {best_run.data.metrics['f1']:.4f}")
print(f"  Accuracy:   {best_run.data.metrics['accuracy']:.4f}")
print(f"  Precision:  {best_run.data.metrics['precision']:.4f}")
print(f"  Recall:     {best_run.data.metrics['recall']:.4f}")

## Step 2: Register the Model

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

MODEL_NAME = "contoso-repair-classifier"

run_model_uri = f"runs:/{best_run.info.run_id}/model"

registered_model = ml_client.models.create_or_update(
    Model(
        name=MODEL_NAME,
        path=run_model_uri,
        type=AssetTypes.MLFLOW_MODEL,
        description=(
            f"Contoso service orders repair-type classifier. "
            f"Trained on service-orders v2. "
            f"Model type: {best_run.data.params.get('model_type', 'N/A')}. "
            f"F1: {best_run.data.metrics['f1']:.4f}"
        ),
        tags={
            "task": "tabular-classification",
            "target": "RepairType",
            "region": "region-1",
            "data_asset": "service-orders",
            "data_version": "2",
            "source_run_id": best_run.info.run_id,
            "f1_score": f"{best_run.data.metrics['f1']:.4f}",
            "stage": "staging",
        },
    )
)

print(f"Registered model: {registered_model.name}")
print(f"  Version: {registered_model.version}")
print(f"  Tags: {registered_model.tags}")

## Step 3: Browse the Full Model Registry

Now the registry has **two models** — the text classifier from Track A
and the tabular classifier from Track B.

In [ ]:
print("=== All registered models ===")
print()

for model_name in ["contoso-lead-classifier", "contoso-repair-classifier"]:
    print(f"Model: {model_name}")
    try:
        for model in ml_client.models.list(name=model_name):
            print(f"  v{model.version} | Task: {model.tags.get('task', 'N/A')} | "
                  f"F1: {model.tags.get('f1_score', 'N/A')} | "
                  f"Data: {model.tags.get('data_asset', 'N/A')} v{model.tags.get('data_version', 'N/A')}")
    except Exception:
        print("  (not yet registered — complete Track A first)")
    print()

In [ ]:
# Trace full lineage for the repair classifier
model_v1 = ml_client.models.get(name=MODEL_NAME, version="1")

print(f"Model: {model_v1.name} v{model_v1.version}")
print(f"  Description: {model_v1.description}")
print(f"\nFull lineage:")
print(f"  Data asset: {model_v1.tags.get('data_asset')} v{model_v1.tags.get('data_version')}")
print(f"  Training run: {model_v1.tags.get('source_run_id')}")
print(f"  F1 at training: {model_v1.tags.get('f1_score')}")

## Go to Azure ML Studio Now

Navigate to **Models** in the left navigation:

1. You should see **two models**: `contoso-lead-classifier` and `contoso-repair-classifier`
2. Click each to compare their tags, linked data assets, and source runs
3. Notice the lineage: each model traces back to different data and experiments

This is what a real team's registry looks like — multiple models, each with full provenance.

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Multi-model registry** | Two models coexist: text classifier + tabular classifier |
| **Same registration API** | Identical `ml_client.models.create_or_update` call |
| **Independent lineage** | Each model links to its own data asset and experiment |
| **Tag-based governance** | Filter by task, region, stage — works across model types |

---

**Next:** [04b - Model Deployment (Tabular)](./04b_model_deployment_tabular.ipynb)